In [ ]:
# JSON 파일 CLIP 파인튜닝용 문장화 & CSV 파일 제작 코드

import json
import os
import pandas as pd
from tqdm import tqdm

# 1. CLIP 학습을 위한 영문 매핑 테이블
MAP = {
    "gender": {"M": "male", "W": "female"},
    "style": {
        "ivy": "casual", "ecology": "casual", "lounge": "casual", "normcore": "casual",
        "hiphop": "streetwear", "grunge": "streetwear",
        "classic": "office wear", "powersuit": "office wear", "metrosexual": "office wear", "cityglam": "office wear",
        "feminine": "feminine", "bodyconscious": "feminine", "lingerie": "feminine",
        "military": "mannish", "bold": "mannish", "genderless": "mannish",
        "sportivecasual": "sporty", "athleisure": "sporty",
        "minimal": "modern minimal",
        "kitsch": "romantic",
        "punk": "hipster", "hippie": "hipster", "popart": "hipster", "space": "hipster",
        "mods": "retro", "disco": "retro", "oriental": "retro"
    },
    "Q2": {1: "spring and autumn", 2: "summer", 3: "winter"},
    "Q3": {1: "commuting", 2: "dating", 3: "formal events", 4: "social gatherings", 5: "daily life", 6: "leisure sports", 7: "traveling", 8: "various occasions"},
    "Q411": {1: "loose", 2: "regular", 3: "tight"},
    "Q412": {1: "dark-toned", 2: "bright-toned"},
    "Q413": {1: "cool-toned", 2: "warm-toned"},
    "Q414": {1: "heavy", 2: "light"},
    "words": {
        "Q4201": "cool", "Q4202": "urban", "Q4203": "trendy", "Q4204": "sophisticated",
        "Q4205": "neat", "Q4206": "fancy", "Q4207": "unique", "Q4208": "plain",
        "Q4209": "open-minded", "Q4210": "practical", "Q4211": "active", "Q4212": "comfortable",
        "Q4213": "cheerful", "Q4214": "feminine", "Q4215": "masculine", "Q4216": "soft"
    }
}

def generate_description(data):
    try:
        item = data['item']
        survey = item['survey']
        
        style_name = MAP['style'].get(item['style'], item['style'])
        gender_name = MAP['gender'].get(item['gender'], 'unisex')
        
        desc = f"A photo of a {item['era']}s {gender_name} {style_name} fashion outfit. "
        desc += f"It is suitable for the {MAP['Q2'].get(survey['Q2'], 'all-year')} season and recommended for {MAP['Q3'].get(survey['Q3'], 'daily life')}. "
        desc += f"The outfit features a {MAP['Q411'].get(survey['Q411'], '')} fit with {MAP['Q412'].get(survey['Q412'], '')} and "
        desc += f"{MAP['Q413'].get(survey['Q413'], '')} colors, creating a {MAP['Q414'].get(survey['Q414'], '')} mood. "
        
        active_words = [MAP['words'][k] for k, v in MAP['words'].items() if survey.get(k, 0) > 0]
        if active_words:
            desc += f"Key characteristics include {', '.join(active_words)} vibes."
            
        return " ".join(desc.split())
    
    except Exception as e:
        return f"Error: {e}"

def main():
    # 📁 1. 경로 설정 (반드시 본인의 컴퓨터 환경에 맞게 수정하세요)
    src_dir = r"H:/data/010.연도별 패션 선호도 파악 및 추천 데이터/01-1.정식개방데이터/huggingface/Training/라벨링데이터/2019남녀통합/2019"    # 원본 JSON 파일이 있는 폴더
    dst_dir = r"H:/data/010.연도별 패션 선호도 파악 및 추천 데이터/01-1.정식개방데이터/huggingface/Training/라벨링데이터/2019남녀통합/문장화"    # 설명이 추가된 JSON 파일이 저장될 폴더
    output_csv = "clip_training_data.csv"       # 최종 생성될 매핑 CSV 파일 이름

    # 저장 폴더가 없으면 생성
    if not os.path.exists(dst_dir):
        os.makedirs(dst_dir)

    json_files = [f for f in os.listdir(src_dir) if f.endswith('.json')]
    print(f"🚀 총 {len(json_files)}개의 데이터 변환 및 매핑을 시작합니다...")

    # 매핑 데이터를 모아둘 빈 리스트
    manifest_data = []

    # 🔄 2. 파일 변환 루프
    for filename in tqdm(json_files, desc="JSON 변환 및 CSV 리스트 생성"):
        try:
            # 원본 JSON 읽기
            with open(os.path.join(src_dir, filename), 'r', encoding='utf-8') as f:
                data = json.load(f)
                
            # 영문 캡션 생성 및 JSON에 추가
            caption = generate_description(data)
            data['description_en'] = caption
            
            # 변환된 JSON 저장
            with open(os.path.join(dst_dir, filename), 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=4)
                
            # 💡 3. CSV 매핑용 데이터 수집
            # JSON에서 이미지 이름 추출 (없으면 파일명 기반으로 생성)
            img_name = data.get('item', {}).get('imgName', filename.replace('.json', '.jpg'))
            
            # 리스트에 사진 이름과 영문 설명 추가
            manifest_data.append({
                "image_path": img_name,
                "caption": caption
            })

        except Exception as e:
            print(f"\n⚠️ [{filename}] 처리 중 오류 발생: {e}")

    # 📊 4. 마스터 CSV 파일 생성
    if manifest_data:
        df = pd.DataFrame(manifest_data)
        df.to_csv(output_csv, index=False, encoding='utf-8-sig')
        print(f"\n🎉 작업 완료! 총 {len(df)}개의 매핑 리스트가 '{output_csv}'에 성공적으로 저장되었습니다.")
    else:
        print("\n❌ CSV에 저장할 데이터가 없습니다.")

if __name__ == "__main__":
    main()

🚀 총 30650개의 데이터 변환 및 매핑을 시작합니다...


JSON 변환 및 CSV 리스트 생성: 100%|██████████| 30650/30650 [00:42<00:00, 725.06it/s]



🎉 작업 완료! 총 30650개의 매핑 리스트가 'clip_training_data.csv'에 성공적으로 저장되었습니다.
